# Deliverable 2 — EDA, Data Leakage & Choosing the Right Model

**ADSC × American Airlines — Fall 2026**

In Deliverable 1 you were handed one model — `XGBRegressor` — and you trained it. It worked, and you got an R² somewhere north of 0.97.

Two questions were never asked:

1. **Was that score real?** (Partly not. Section 3.)
2. **Was XGBoost the right choice?** (You have no idea, because you never tried anything else.)

This deliverable answers both. You will go deeper on EDA, learn what **data leakage** is and measure what it cost you, then run a genuine model-selection exercise: **Lasso vs Ridge**, **Random Forest vs XGBoost**, and **ARIMA** — plus the choice that sits underneath all of them and that nobody teaches you to make deliberately, your **loss function**.

---

## What you will learn

| Section | Topic | The question it answers |
|---|---|---|
| **2** | **Deeper EDA** | What does the data actually look like, and what does that imply about which model to reach for? |
| **3** | **Data leakage** | Why was the Deliverable 1 score too good, and how do I stop it happening again? |
| **4** | **Lasso vs Ridge** | Two linear models, two penalties. When do I want shrinkage, and when do I want feature selection? |
| **5** | **Loss functions** | MSE, MAE, Huber, quantile — what am I actually asking the model to minimise, and does it match what the business cares about? |
| **6** | **XGBoost vs Random Forest vs ARIMA** | Three model families. Which fits *this* problem, and why is one of them answering a completely different question? |
| **7** | **Pick one and defend it** | Given everything above, what would you ship to American Airlines? |

The single most important idea in here: **model selection is not a leaderboard.** It is a set of choices — model family, loss function, regularisation — each of which should follow from something you observed in the data or something the business told you. By the end you should be able to justify every one of your choices without using the phrase "it scored the best".

---

## What we are building on from Deliverable 1

We assume all of this and will not re-explain it:

- loading the CSV, `.shape` / `.info()` / `.describe()`
- null counts, mean / median / std / quartiles / IQR with NumPy
- the 1.5 × IQR outlier rule
- the price histogram, price-by-airline box plot, duration-vs-price scatter
- `LabelEncoder`, an 80/20 `train_test_split`, `XGBRegressor`
- MAE, RMSE, R², and gain-based feature importance

Keep **your own completed Deliverable 1 notebook** open in another tab — it is in `deliverable1/`, saved under your name — because we refer back to it by section number throughout.

---

## The git workflow changed — read this before you touch anything

Deliverable 1 was solo: you cut `deliverable1/<your-name>` and opened a PR straight into `main`. With 70+ members that does not scale, and it is not how real teams work either.

**From Deliverable 2 onward you work in a team, and your team owns a branch tree.**

```
main                          <- protected. Only a PM merges here.
 └── team-07/main             <- YOUR TEAM's integration branch ("team main")
      ├── team-07/alex-chen    <- a member branch: your own work
      ├── team-07/priya-r      <- another member, in parallel
      └── team-07/sam-lee
```

Three rules, and they are the whole model:

1. **Your team's trunk is `team-##/main`.** Replace `##` with the team number your PM assigned you — zero-padded, so team 7 is `team-07`, team 12 is `team-12`.
2. **You personally work on `team-##/<your-name>`.** Never commit directly to `team-##/main`.
3. **Work moves up the tree:** `team-##/<your-name>` → `team-##/main` → `main`. Your PR goes into **`team-##/main`**, reviewed by a teammate. Only `team-##/main` ever opens a PR into `main`, and only a PM merges that one.

### Setting it up

**One person on the team** creates the team trunk (do this once, then tell your teammates it exists):

```sh
git switch main
git pull origin main
git switch -c team-07/main          # use YOUR team number
git push -u origin team-07/main
```

**Every member** then branches off the team trunk — not off `main`:

```sh
git fetch origin
git switch -c team-07/alex-chen origin/team-07/main
git push -u origin team-07/alex-chen
```

### Submitting

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push

# then on GitHub: open a PR from team-07/alex-chen  ->  team-07/main
# (base branch is your TEAM's main, NOT the repo's main)
```

> **⚠️ The one naming rule that will bite you.** Git stores branches as files, so no branch name may be a *prefix* of another inside the same namespace. Because `team-07/main` exists, `team-07/main/fix` is **impossible** — you will get `cannot lock ref ... 'team-07/main' exists`. Keep every name a flat leaf: `team-07/alex-chen`, `team-07/alex-chen-v2`. Never nest.

> **⚠️ Notebooks and merge conflicts.** A `.ipynb` is JSON, and git conflicts on it are genuinely horrible. Two habits make it survivable:
> - **Clear outputs before every commit** (Kernel → Restart & Clear Output). Most notebook conflicts are rendered output, not code.
> - **Each member completes their own copy of this notebook.** You are *not* meant to split the cells across teammates. The notebook is individual learning; the shared team artifact is `deliverable2/TEAM_REPORT.md`, which your team fills in together on `team-##/main`.

Full detail, including each member's named review duty, is in [`deliverable2/README.md`](README.md). The general workflow rules have not changed — [`CONTRIBUTING.md`](../CONTRIBUTING.md) is still the operating manual.

---

> ### 🗒️ A note on AI tools
>
> Half of this deliverable is *judgment*: is this leakage, is Lasso the right penalty, does ARIMA even apply here. An AI will hand you a confident answer to every one of those, and you will have learned nothing and be unable to defend it when a teammate asks in review.
>
> **Use AI as a tutor, not an author:**
> - "Why does Lasso drive coefficients to exactly zero when Ridge does not?"
> - "What does `objective='reg:pseudohubererror'` change about training?"
> - "Why is ARIMA inappropriate for data without a time index?"
>
> **You must be able to explain every line you submit.** A teammate *will* review your PR and *will* ask.

---

## How to approach this deliverable

Read this before you start. It will save you more time than it costs.

### 1. The written answers are the deliverable — not the plots

Every section ends with a markdown prompt asking *why*. Those are the thing being assessed. A plot with no conclusion attached is worth nothing, and a number with no interpretation is worth less than no number at all, because it invites someone to act on it.

Concretely: when a cell tells you to make a histogram, the deliverable is not the histogram. It is the sentence **"...therefore I should \_\_\_ when I model."** If you cannot finish that sentence, you have not finished the cell.

### 2. Expect your R² to go down, and treat that as success

Section 3 will show you that your Deliverable 1 score was inflated. Your honest number will be lower. **That is the correct outcome.** Lowering your own score on purpose, and being able to explain why the new number is the real one, is the single most professional thing a data scientist does. Do not try to tune your way back up to 0.98 — you would only be recreating the bug.

### 3. How to think about choosing a model

The instinct from Deliverable 1 is "run them all, keep the best score". Resist it. Work through the questions in this order, because each one can eliminate options before you spend compute on them:

1. **What shape is my data, and what shape does this model require?** This is first because it is disqualifying. ARIMA needs an ordered time series; if you do not have one, no amount of tuning helps. Section 6 is built around this question.
2. **What am I asking the model to minimise?** The default is squared error, which is a choice you have been making silently. Section 5 is about making it on purpose.
3. **What does the stakeholder need?** A client who needs to *explain* a price to a customer may be better served by a Ridge model with readable coefficients than by a boosted ensemble that scores two points higher. Accuracy is one requirement among several.
4. **What does it cost?** Fit time, tuning burden, how many ways it can break in production. A model that takes 40× longer for 1% more R² is usually the wrong call.
5. **Only then: what scores best?**

If your final justification is "it had the highest R²", you have skipped the deliverable.

### 4. How to work the code

- **Run cells in order, top to bottom.** Later sections depend on variables defined earlier (`clean`, `X_train`, `prep_tree`). If something is `None` or undefined, you skipped a cell.
- **Check your work as you go.** After any transform, look at the result — `.head()`, `.shape`, `.value_counts()`. Most bugs in data code are silent: the wrong answer computes perfectly happily.
- **Some cells are slow.** Random Forest on 240k rows and 5-fold CV both take minutes. If a cell is painful on your machine, **subsample it and say so in your PR** — that is a normal engineering decision, not cheating. `X.sample(80_000, random_state=RANDOM_STATE)` is your friend.
- **Read the error, then the line above it.** Python tracebacks name the failing line. Nine times in ten the actual mistake is one line earlier.
- **Commit per section**, not in one lump at the end. Small commits are the entire reason your history is useful to a reviewer.

### 5. How to work with your team

Each member completes **their own full copy** of this notebook — the work is individual, because the learning is individual. What your team does **together** is agree on the conclusions and write them into `TEAM_REPORT.md` on `team-##/main`.

Beyond that, how your team organises itself is up to your team. Decide in your first meeting and write the decision down somewhere your PM can see it.

One piece of hard-won advice: **clear your outputs before every commit** (Kernel → Restart & Clear Output). Notebook merge conflicts are mostly conflicts in rendered output, not in code, and this one habit prevents the large majority of them.

---

## References

Every link below was checked and goes to the specific page you need — not a search result or a landing page. Keep the ones for your current section open in browser tabs.

### Start here

| | |
|---|---|
| **Dataset** — Flight Price Prediction (EaseMyTrip) | https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction |
| **scikit-learn: Common pitfalls and recommended practices** — the single most useful page for this deliverable. Shows a "Wrong" and a "Right" version of each mistake. | https://scikit-learn.org/stable/common_pitfalls.html |
| **An Introduction to Statistical Learning** — the standard undergraduate text, **free PDF**. Ch. 6 covers Ridge/Lasso, Ch. 8 covers trees and boosting. Python edition (ISLP) also available. | https://www.statlearning.com/ |

### Section 3 — leakage & validation

| | |
|---|---|
| Data leakage, with a worked example of a score collapsing from 0.76 to chance once the leak is fixed | https://scikit-learn.org/stable/common_pitfalls.html#data-leakage |
| Inconsistent preprocessing — why the transform must be fitted once and applied, not refitted | https://scikit-learn.org/stable/common_pitfalls.html#inconsistent-preprocessing |
| Pipelines and composite estimators (`Pipeline`, `ColumnTransformer`) | https://scikit-learn.org/stable/modules/compose.html |
| Cross-validation: evaluating estimator performance | https://scikit-learn.org/stable/modules/cross_validation.html |

### Section 4 — Lasso vs Ridge

| | |
|---|---|
| Ridge regression — §1.1.2 of the Linear Models guide | https://scikit-learn.org/stable/modules/linear_model.html#ridge-regression-and-classification |
| Lasso — §1.1.3, including the note on why the objective is scaled by `1/(2·n_samples)` | https://scikit-learn.org/stable/modules/linear_model.html#lasso |
| Elastic-Net — §1.1.5 | https://scikit-learn.org/stable/modules/linear_model.html#elastic-net |
| **Common pitfalls in interpreting linear coefficients** — worked example of why scale matters, why coefficients are conditional not marginal, and how correlated features make coefficients unstable. Directly relevant to reading your Lasso output. | https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html |

### Section 5 — loss functions

| | |
|---|---|
| XGBoost learning-task parameters — the full `objective` list (`reg:squarederror`, `reg:absoluteerror`, `reg:pseudohubererror`, `reg:quantileerror`) | https://xgboost.readthedocs.io/en/stable/parameter.html#learning-task-parameters |
| **Prediction intervals for gradient boosting regression** — the reference implementation for Section 5's quantile models, including the coverage check. Note its own conclusion: *each model performs best on the metric it was trained for.* | https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_quantile.html |
| Regression metrics — MAE, RMSE, R², and `mean_pinball_loss` | https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics |

### Section 6 — model families

| | |
|---|---|
| **Introduction to Boosted Trees** — the clearest explanation of additive training anywhere, and it explicitly notes that random forests and boosted trees share the same model form and differ in *how they are trained*. Read this before writing your bagging-vs-boosting answer. | https://xgboost.readthedocs.io/en/stable/tutorials/model.html |
| Random forests and randomized tree ensembles — §1.11.2 | https://scikit-learn.org/stable/modules/ensemble.html#random-forests |
| Gradient-boosted trees — §1.11.1 | https://scikit-learn.org/stable/modules/ensemble.html#gradient-boosted-trees |
| `statsmodels` ARIMA API — the `order=(p, d, q)` signature | https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html |
| **Forecasting: Principles and Practice, Ch. 9 — ARIMA models.** Free online, and the standard reference. | https://otexts.com/fpp3/arima.html |
| §9.1 Stationarity and differencing — read this before choosing your `d` | https://otexts.com/fpp3/arima.html#sec-stationarity |

---

## Setup

Copy the dataset you already downloaded to **`deliverable2/data/Clean_Dataset.csv`**. That path is git-ignored; **do not commit the CSV.**

One new package this time — `statsmodels`, for ARIMA:

```sh
pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels
```

---
# Section 0 — Imports & helpers

Run this cell first. Everything below depends on it.

Some cells in this notebook are **given to you complete** — the helpers, and the reproduction of your Deliverable 1 baseline. Read them; they are the reference implementations you will copy patterns from. The cells marked `# TODO` are yours.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- splitting & validation ------------------------------------------------
from sklearn.model_selection import train_test_split, KFold, cross_validate

# --- preprocessing ---------------------------------------------------------
# OneHotEncoder  : one 0/1 column per category. No invented ordering.
# OrdinalEncoder : integers in an order YOU specify (zero < one < two_or_more).
# StandardScaler : centres and scales numerics. Optional for trees,
#                  MANDATORY for Lasso/Ridge -- Section 4 explains why.
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# --- composition -----------------------------------------------------------
# Pipeline guarantees every transform is fitted on the TRAINING fold only.
# That is our main defence against the leakage in Section 3.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- the models we will compare -------------------------------------------
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# --- metrics ---------------------------------------------------------------
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_pinball_loss)

# --- display ---------------------------------------------------------------
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42   # every split, every model. Reproducibility is not optional.
DATA_PATH = 'deliverable2/data/Clean_Dataset.csv'   # relative to the repo root

print('pandas', pd.__version__, '| numpy', np.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)
import xgboost; print('xgboost', xgboost.__version__)

# statsmodels is only needed in Section 6c (ARIMA). If this import fails,
# `pip install statsmodels` and restart the kernel.
import statsmodels.api as sm
print('statsmodels', sm.__version__)

In [ ]:
def regression_report(y_true, y_pred, label=''):
    """Return a dict of the four metrics we use everywhere in this notebook.

    MAE   -- average absolute miss, in rupees. Interpretable.
    RMSE  -- squared-error based, so large misses dominate. RMSE >> MAE means a
             few very bad predictions are hiding behind a decent average.
    R2    -- fraction of price variance explained.
    MAPE% -- average miss as a PERCENTAGE of true price. The business metric: being
             off by 2,000 INR is trivial on a 52,000 ticket and awful on a 6,000 one.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'model': label,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': r2_score(y_true, y_pred),
        'MAPE%': float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
    }


# Running scoreboard. Every model we fit goes on here, so at the end we can read
# the whole deliverable as one comparison table.
SCOREBOARD = []

def record(result):
    """Append a regression_report dict to the scoreboard and show the table."""
    SCOREBOARD.append(result)
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)


def show_scoreboard():
    return pd.DataFrame(SCOREBOARD).set_index('model').round(4)

print('Helpers defined: regression_report, record, show_scoreboard')

---
# Section 1 — Load the data

Nothing new here conceptually — but note the two given lines after the load, and read the comment attached to them.

In [ ]:
# TODO: Load the CSV into `raw`, print its shape, and print raw.columns.tolist().
raw = None  # TODO

In [ ]:
# Two housekeeping fixes, given to you.
#
# 1. The CSV as published ships a nameless first column that pandas labels
#    'Unnamed: 0'. It is the row number from whoever exported the file -- somebody
#    ran to_csv() without index=False. It is NOT a feature: a flight you are asked
#    to price next week has no row number in your training file. Drop it.
#    (Deliverable 1 never dropped it, which means it was sitting in your X. Section 3
#    measures what that cost you.)
#
# 2. The Deliverable 1 schema table used 'Source City' / 'Price'; the actual CSV
#    uses 'source_city' / 'price'. Documentation drifting from the file is routine.
#    We lowercase everything so the rest of the notebook works either way.

df = raw.copy()
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df = df.drop(columns=[c for c in df.columns if c.startswith('unnamed')])

print('shape:', df.shape)
print('columns:', df.columns.tolist())
df.head(3)

---
# Section 2 — EDA that informs a modelling decision

**Goal:** go deeper than Deliverable 1, and tie every observation to a modelling consequence.

Deliverable 1's EDA was descriptive: here is the distribution, here are the categories, here is a scatter plot. Useful, but it stopped short. **EDA earns its keep when it changes what you do next.** So for every plot in this section, the deliverable is not the plot — it is the sentence "...therefore I should \_\_\_ when I model".

Four things we need to know before choosing a model:

| # | What we look at | The modelling decision it drives |
|---|---|---|
| 2a | How `price` is distributed | Which **loss function** (Section 5) and whether to transform the target |
| 2b | Whether relationships are **linear** | Whether a linear model (Section 4) can compete at all |
| 2c | The **booking curve** over `days_left` | Whether a **time-series** model (Section 6) has anything to work with |
| 2d | Category counts and dominance | Which **encoding**, and whether one feature will swamp everything |

---

### 2a — The shape of the target

Deliverable 1 plotted the price histogram and noted it was right-skewed. Go further: plot it **split by class**.

In [ ]:
# TODO:
#   1. Histogram of price, split by class, on one axes:
#        sns.histplot(data=df, x='price', hue='class', bins=80, element='step')
#
#   2. The same thing on a log x-axis -- add plt.xscale('log'). Does the shape become
#      more symmetric? That is a direct hint about whether to model log(price).
#
#   3. Compute skewness for the raw and the log-transformed target:
#        df['price'].skew(), np.log1p(df['price']).skew()
#      Skew near 0 is symmetric; above ~1 is strongly right-tailed.
#
#   4. Mean and median price per class:
#        df.groupby('class')['price'].agg(['count', 'mean', 'median', 'std'])

**What you should see, and what it means.** *(3–4 sentences.)*

The distribution is not one skewed hump — it is **two separate populations**, Economy and Business, with Business roughly an order of magnitude more expensive.

That matters for Section 5 in a concrete way. Squared error (MSE) penalises a ₹5,000 miss identically whether the ticket costs ₹6,000 or ₹52,000 — but the first is an 83% error and the second is under 10%. So **a model trained on MSE will quietly prioritise getting Business right**, because that is where the big absolute numbers are.

Write down: the two class means, the raw skew, the log skew, and one sentence on which loss function you now expect to prefer and why.

*(Write here)*

### 2b — Is this problem linear?

Before you can argue that you need a boosted tree ensemble, you have to know what a straight line gets you. And before you can read a correlation matrix, note the limit of the tool: **Pearson correlation measures only the *linear* part of a relationship.** A perfect U-shaped relationship has a correlation near zero. A low correlation does not mean "no relationship" — it means "no *linear* relationship", which is a much narrower claim.

In [ ]:
# TODO:
#   1. Correlation heatmap of the numeric columns:
#        num = df[['duration', 'days_left', 'price']]
#        sns.heatmap(num.corr(), annot=True, cmap='coolwarm', center=0)
#
#   2. Now add the single strongest categorical as a number, to see how it compares:
#        tmp = num.copy()
#        tmp['is_business'] = (df['class'] == 'Business').astype(int)
#        sns.heatmap(tmp.corr(), annot=True, cmap='coolwarm', center=0)
#
#   3. Scatter price vs duration with a linear fit on top, to see the fit fail:
#        s = df.sample(5000, random_state=RANDOM_STATE)
#        sns.regplot(data=s, x='duration', y='price',
#                    scatter_kws={'s': 4, 'alpha': 0.3},
#                    line_kws={'color': 'red'})
#
# Then answer: is `is_business` more correlated with price than either real numeric
# feature? What does a single straight line through the duration/price cloud miss?

**Your read.** *(3–4 sentences. Which feature carries the most linear signal? Where does the straight line visibly fail? Does the `duration` correlation being weak prove duration does not matter — and if not, why not?)*

*(Write here)*

### 2c — The booking curve: the one piece of this dataset with a time axis

This is the most important plot in the notebook, and it is the one Deliverable 1 never made.

`days_left` is **days until departure**. It is not a timestamp, but it *is* an ordered horizon — so if you average price at each value of `days_left`, you get a genuine **sequence**: how fare moves as departure approaches. Airlines call this the booking curve, and forecasting it is a real job.

That ordered sequence is the only thing in this dataset a time-series model like ARIMA can operate on. Section 6c comes back to it.

In [ ]:
# TODO:
#   1. Build and plot the curve. Note the x-axis is REVERSED, so that time flows
#      left-to-right toward departure:
#        curve = df.groupby('days_left')['price'].agg(['mean', 'median', 'count'])
#        plt.plot(curve.index, curve['mean'], marker='o', ms=3)
#        plt.gca().invert_xaxis()
#        plt.xlabel('days until departure  (departure is at the right)')
#        plt.ylabel('mean price (INR)')
#
#   2. Same plot, one line per class. The two cabins behave very differently:
#        for cabin, grp in df.groupby('class'):
#            c = grp.groupby('days_left')['price'].mean()
#            plt.plot(c.index, c.values, marker='o', ms=3, label=cabin)
#        plt.gca().invert_xaxis(); plt.legend()
#
#   3. How many points are in this series? len(curve). Keep that number in mind --
#      it is the entire sample size available to ARIMA in Section 6c.
#
# Then answer below.

**Read the curve.** *(4–5 sentences.)*

- Is the relationship between `days_left` and price **linear**? At roughly what point does it turn?
- Economy vs Business: same shape, or different?
- This answers something important about Section 4. A linear model given raw `days_left` fits **one straight line** through this curve. Looking at the shape, how badly will that go — and what could you do to a linear model to let it cope? *(Think: what if `days_left` were bucketed, or log-transformed, or squared?)*
- How many data points does the aggregate curve have? Is that a lot or a little for fitting a time-series model?

*(Write here)*

### 2d — Categories: counts, dominance, and encoding

In [ ]:
# TODO:
#   1. For each categorical column, print the number of unique values:
#        cats = ['airline', 'source_city', 'departure_time', 'stops',
#                'arrival_time', 'destination_city', 'class']
#        for c in cats:
#            print(f'{c:20s} {df[c].nunique():3d}  {sorted(df[c].unique())}')
#
#      You need those exact spellings in Section 3 -- copy them down.
#
#   2. Mean price by class AND airline together, to see the interaction:
#        pd.crosstab(df['airline'], df['class'], values=df['price'],
#                    aggfunc='mean')
#      Do all airlines even sell Business?
#
#   3. Mean price by route, top and bottom 5:
#        route_price = df.groupby(['source_city', 'destination_city'])['price'].mean()
#        route_price.sort_values().head(5), route_price.sort_values().tail(5)

> **Which columns are ordered, and which are not?** This matters in Section 4 more than you would guess.
>
> - `stops` is **ordinal**: `zero < one < two_or_more`. That is a real ranking.
> - `departure_time` / `arrival_time` are **ordinal**: `Early_Morning < Morning < Afternoon < Evening < Night < Late_Night` — they are a clock.
> - `airline`, `source_city`, `destination_city` are **nominal**. There is no sense in which Vistara > SpiceJet.
>
> Deliverable 1 used `LabelEncoder` on all seven, which assigns integers **alphabetically**. For `stops` that gives `one=0, two_or_more=1, zero=2` — telling the model a direct flight ranks *above* a two-stop flight. Not just arbitrary: backwards.
>
> A tree can route around this by spending extra splits to separate the categories again, so the cost shows up as wasted capacity. **A linear model cannot route around it at all** — it gets one coefficient per column and is forced to assume evenly-spaced, ordered effects. Section 4 shows you exactly how much that costs.

### 2e — EDA summary: four decisions

Fill this in before you model. This is the deliverable for Section 2 — not the plots.

| Observation | Therefore, when I model, I will... |
|---|---|
| Target distribution is... | |
| The strongest single predictor appears to be... | |
| The `days_left` relationship is... | |
| Ordered categories are... | |

---
# Section 3 — Data leakage

**Goal:** understand why the Deliverable 1 score was too good, fix it, and measure the cost of each fix.

> **Leakage** is when information reaches the model at training time that will not be available at prediction time.

It is not exotic. It is the default result of writing code in the order that feels natural — "clean everything, then split" — because every transform fitted before the split has already seen the test set.

Deliverable 1 had three leaks:

| # | Leak | Where |
|---|---|---|
| 1 | **The row counter.** `Unnamed: 0` was never dropped, so it sat in `X`. A future flight has no row number. | D1 §3a |
| 2 | **Duplicate rows across the split.** Once the unique counter is removed, this dataset contains repeated rows; copies can land in train *and* test, so the model is graded on rows it memorised. | D1 §3c |
| 3 | **Encoders fitted on the full dataset.** `df[col] = le.fit_transform(df[col])` ran in §3b, *before* the split in §3c. | D1 §3b |

**Be precise about leak 3's severity**, because overstating it is its own error. A plain `LabelEncoder` on complete categorical columns learns a list of category *names* — it learns nothing about price, so the practical damage is small. It is still **wrong as a habit**, because the identical code shape is severe the moment the transform learns something numeric:

- a `StandardScaler` fitted pre-split leaks the test set's mean and variance
- a **target encoder** fitted pre-split leaks the test set's *prices* directly
- an imputer fitted pre-split leaks the test distribution

You will not remember which transforms are safe at 2am. **Fit every transform inside the split, always.** That is what `Pipeline` is for.

### 3a — Reproduce the Deliverable 1 baseline (given)

Given to you complete, so every member starts from the identical reference number. Read it and match each line to a section of the old notebook.

In [ ]:
def deliverable1_baseline(path, random_state=RANDOM_STATE):
    """Reproduce Deliverable 1 exactly -- leaks and all.

      D1 S1  : read_csv
      D1 S2a : dropna
      D1 S3a : drop 'flight'        <- but NOT the row counter
      D1 S3b : LabelEncoder on the FULL frame, before the split   <- leak 3
      D1 S3c : X = everything except price; 80/20 split           <- dupes straddle it
      D1 S4  : XGBRegressor, same hyperparameters
    """
    from sklearn.preprocessing import LabelEncoder

    frame = pd.read_csv(path)
    frame.columns = [c.strip().lower().replace(' ', '_') for c in frame.columns]
    frame = frame.dropna().drop(columns=['flight'])

    for col in ['airline', 'source_city', 'departure_time', 'stops',
                'arrival_time', 'destination_city', 'class']:
        # Fitted on 100% of rows -- including the ones we will call "unseen".
        frame[col] = LabelEncoder().fit_transform(frame[col])

    X = frame.drop(columns=['price'])
    y = frame['price']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=random_state)

    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=random_state, n_jobs=-1)
    model.fit(X_tr, y_tr)
    return model, X_tr, X_te, y_tr, y_te, model.predict(X_te)


t0 = time.time()
d1_model, d1_X_tr, d1_X_te, d1_y_tr, d1_y_te, d1_pred = deliverable1_baseline(DATA_PATH)
print(f'trained in {time.time() - t0:.1f}s')
print('features the D1 model was given:', d1_X_tr.columns.tolist())

record(regression_report(d1_y_te, d1_pred, 'D1 baseline (3 leaks)'))

In [ ]:
# TODO: Where did the row counter rank in that model's importances?
#
#   pd.Series(d1_model.feature_importances_, index=d1_X_tr.columns) \
#     .sort_values(ascending=False)
#
# Anything above last place is a problem: it means the model is partly pricing
# flights off the order rows happen to sit in the file.
#
# Then check WHY the row number carries signal at all -- the file is sorted:
#   raw_sorted = pd.read_csv(DATA_PATH)
#   raw_sorted.columns = [c.strip().lower().replace(' ', '_')
#                         for c in raw_sorted.columns]
#   raw_sorted.groupby('class')['unnamed:_0'].agg(['min', 'max', 'count'])
#
# (Adjust that column name to whatever your printout in Section 1 showed.)

### 3b — Remove the leaks, one at a time

One change per step, so each result is **attributable**. Change three things at once and you learn nothing about any of them.

In [ ]:
# TODO: Step 1 -- the D1 baseline with the row counter dropped.
#
# Copy the body of deliverable1_baseline and drop the 'unnamed:_0' column too.
# Keep everything else identical -- same random_state, same hyperparameters -- so
# the comparison is clean.
#
#   record(regression_report(y_te, pred, 'no row counter'))

In [ ]:
# TODO: Step 2 -- also remove duplicate rows, BEFORE the split.
#
# Order is the entire point: dedupe FIRST, then split. If you split first and dedupe
# each side separately, copies still straddle the boundary.
#
#   deduped = frame.drop_duplicates()
#   print('dropped', len(frame) - len(deduped), 'duplicate rows')
#
#   record(regression_report(y_te, pred, 'no counter + deduped'))
#
# Expect the largest single drop here. Before you move on, say why: which rows can
# the model no longer memorise?

### 3c — Step 3: a `Pipeline`, so leakage becomes structurally impossible

A **`Pipeline`** is a list of `(name, transform)` steps ending in a model, wrapped to behave as one estimator. What matters is what `.fit()` does:

```
pipe.fit(X_train, y_train)
   -> every transform learns ONLY from X_train
pipe.predict(X_test)
   -> every transform APPLIES what it learned; it never re-learns from X_test
```

You cannot accidentally fit on the test set, because you never call `.fit` on a transform yourself. And this becomes essential rather than merely tidy in Section 4, where we cross-validate over many `alpha` values: without a pipeline, the scaler would be refitted on each fold's test data and every number would be slightly wrong.

A **`ColumnTransformer`** routes different columns to different transforms:

```python
ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
    ('ord', OrdinalEncoder(categories=[...]),       ordinal_cols),
    ('num', StandardScaler(),                       numeric_cols),
])
```

`handle_unknown='ignore'` is the production-mindset flag: if an unseen category appears at prediction time (a new airline launches), encode it as all-zeros instead of crashing.

In [ ]:
# TODO: Define the column groups and the explicit category orders.
#
# NOMINAL (one-hot; no order exists):
#   'airline', 'source_city', 'destination_city'
# ORDINAL (integer-encode IN ORDER):
#   'stops', 'departure_time', 'arrival_time'
# NUMERIC:
#   'duration', 'days_left'
#
# Use the exact spellings you printed in Section 2d. A `categories=` list that does
# not match the data will raise at fit time.

NOMINAL = None      # TODO
ORDINAL = None      # TODO
NUMERIC = None      # TODO

STOPS_ORDER = None  # TODO: ['zero', 'one', 'two_or_more']
TIME_ORDER  = None  # TODO: the six time bands, earliest to latest

In [ ]:
# TODO: Build TWO preprocessors. You need both, and the reason is Section 4.
#
# `prep_tree` -- for Random Forest / XGBoost. Scaling is irrelevant to trees
#                (a split threshold does not care about units), so skip it.
#
#   prep_tree = ColumnTransformer([
#       ('nom', OneHotEncoder(handle_unknown='ignore'), NOMINAL),
#       ('ord', OrdinalEncoder(categories=[STOPS_ORDER, TIME_ORDER, TIME_ORDER],
#                              handle_unknown='use_encoded_value',
#                              unknown_value=-1), ORDINAL),
#       ('num', 'passthrough', NUMERIC),
#   ], remainder='drop')
#
# `prep_linear` -- for Lasso / Ridge. Identical, except numerics are SCALED.
#                  Section 4 explains why this is mandatory, not optional.
#
#   prep_linear = ColumnTransformer([
#       ... same first two ...
#       ('num', StandardScaler(), NUMERIC),
#   ], remainder='drop')
#
# remainder='drop' is what keeps `flight` out of the model without you having to
# delete the column from the DataFrame.

prep_tree = None    # TODO
prep_linear = None  # TODO

In [ ]:
# TODO: Build the clean dataset and the one split that every model in Sections 4-6
# will share. Using one split for all models is what makes the comparison fair.
#
#   clean = df.drop_duplicates().reset_index(drop=True)
#   X = clean.drop(columns=['price'])
#   y = clean['price']
#   X_train, X_test, y_train, y_test = train_test_split(
#       X, y, test_size=0.2, random_state=RANDOM_STATE)
#
#   print(f'{len(X_train):,} train / {len(X_test):,} test')
#
# Then fit an XGBoost pipeline on it -- this is your honest baseline:
#   xgb_pipe = Pipeline([('prep', prep_tree),
#                        ('model', XGBRegressor(n_estimators=300, max_depth=6,
#                                               learning_rate=0.1, subsample=0.8,
#                                               colsample_bytree=0.8,
#                                               random_state=RANDOM_STATE,
#                                               n_jobs=-1))])
#   xgb_pipe.fit(X_train, y_train)
#   record(regression_report(y_test, xgb_pipe.predict(X_test), 'XGBoost (no leaks)'))

clean = None  # TODO
X = None      # TODO
y = None      # TODO

### 3d — Target leakage: the expensive kind

The three leaks above were about *process*. This one is about a *feature*, and it is the one that ends projects.

Here is a feature a well-meaning teammate will propose:

```python
df['price_per_hour'] = df['price'] / df['duration']
```

It is intuitive, it is a real industry unit (yield per block hour), and adding it will take your R² to something like 0.999.

**It is computed from the target.** At prediction time you do not have `price` — predicting it is the whole job. The feature can only exist for rows where you already know the answer.

**Target leakage** is the most expensive mistake in applied ML precisely because it does not look like a bug. Nothing crashes, no warning fires, your metrics improve, and the model is worthless.

> **The test, five seconds per feature:** *"at the moment I need a prediction, do I have this value?"* If the honest answer is "no, I would need the price first" — delete it.

In [ ]:
# TODO: Prove it to yourself, then delete it. Feeling how good the fake number
# looks is the point.
#
#   leaky = clean.copy()
#   leaky['price_per_hour'] = leaky['price'] / leaky['duration']
#   Xl = leaky[['price_per_hour', 'duration', 'days_left']]
#   yl = leaky['price']
#   Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.2,
#                                                 random_state=RANDOM_STATE)
#   m = XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
#   m.fit(Xl_tr, yl_tr)
#   print(regression_report(yl_te, m.predict(Xl_te), 'TARGET LEAK -- do not ship'))
#
# Do NOT record() this. A known-invalid number does not belong in your comparison
# table -- that is exactly how a fake score ends up in a slide deck three weeks later.

### 3e — One split is an anecdote

Even leak-free, our score rests on **one** arbitrary 80/20 split. Change `random_state` and the number moves. How much is measurable, and you should measure it before quoting anything.

**K-fold cross-validation** splits the data into k parts, trains k times, validates on each part once. The **mean** is a better estimate; the **standard deviation** tells you how much your single number could have been luck.

> Quote scores as **mean ± std**. A model at 0.95 ± 0.002 and one at 0.95 ± 0.04 are not the same model, and the second is not ready to ship.

In [ ]:
# TODO: 5-fold CV on your XGBoost pipeline.
#
#   cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
#   scores = cross_validate(xgb_pipe, X, y, cv=cv,
#                           scoring=['r2', 'neg_mean_absolute_error'],
#                           n_jobs=1, return_train_score=True)
#
# Print mean, std and min/max across folds for both metrics, then compare
# train_r2 against test_r2 -- a wide gap is the signature of overfitting.
#
# TIMING: 5 XGBoost fits on ~300k rows. Budget 1-3 minutes. If that is painful,
# subsample for the CV cells only and SAY SO in your PR:
#   Xs = X.sample(80_000, random_state=RANDOM_STATE); ys = y.loc[Xs.index]
# Keep n_jobs=1 in cross_validate -- XGBoost already uses all cores, and nesting
# the two makes them fight over CPU.

**The leak ledger.** *(4–6 sentences.)*

- Which single fix cost the most R²? Why that one?
- Your honest number is lower than your Deliverable 1 number. **Which would you put in a report to American Airlines, and how would you explain the difference to someone who already saw the first one?**
- What does your CV standard deviation tell you about how much to trust one decimal place of these scores?

*(Write here)*

---
# Section 4 — Lasso vs Ridge: two ways to regularise a linear model

**Goal:** understand what regularisation *is*, what the L1 and L2 penalties do differently, and why that difference is useful rather than academic.

### Why fit a linear model at all, when you already have XGBoost?

Three reasons, and none of them is "it might win":

1. **It is a reference point.** "XGBoost got R² 0.95" is meaningless until you know a linear model gets 0.90 — or 0.55.
2. **It is interpretable.** A linear model hands you one coefficient per feature: *this airline adds ₹1,800 to the fare, all else equal.* No tree ensemble gives you a sentence like that, and that sentence is often what a client actually wants.
3. **Lasso does feature selection for free**, which is the real payoff in this section.

### The shared objective

Both models minimise squared error **plus** a penalty on the size of the coefficients:

```
Ridge  (L2):   minimise   ||y - Xw||²  +  α · Σ wⱼ²
Lasso  (L1):   minimise   ||y - Xw||²  +  α · Σ |wⱼ|
```

The only difference is `wⱼ²` versus `|wⱼ|`. That one change produces completely different behaviour:

| | **Ridge (L2)** | **Lasso (L1)** |
|---|---|---|
| Penalty | sum of **squared** coefficients | sum of **absolute** coefficients |
| Shrinks coefficients | yes, smoothly toward zero | yes |
| Sets them to **exactly zero** | essentially never | **yes** — that is the point |
| Correlated features | **shares** the weight between them | **picks one**, zeroes the rest |
| Gives you | a stable model using every feature | a **sparse** model, i.e. feature selection |
| Closed-form solution | yes | no — needs iterative optimisation |
| Reach for it when | you believe most features matter a little | you believe few features matter a lot |

### Why does L1 produce exact zeros and L2 does not?

The geometric picture is the one worth remembering. Think of the penalty as a budget constraining where `w` can live:

- The **L2** budget `Σwⱼ² ≤ t` is a **circle** (sphere). Smooth, no corners.
- The **L1** budget `Σ|wⱼ| ≤ t` is a **diamond**, and its corners sit **on the axes**.

The solution lands where the error contours first touch that region. A diamond's corners stick out, so contours tend to touch at a corner — and a corner *is* a point where some coefficient equals exactly zero. A circle has no corners, so the touch point almost always has every coefficient slightly non-zero.

> **⚠️ Scaling is mandatory here, not optional.** Both penalties sum over coefficient *magnitudes*, and a coefficient's magnitude depends on its feature's units. `duration` in hours (0–50) and `days_left` in days (1–49) are comparable, but if one were in minutes its coefficient would be ~60× smaller and the penalty would barely touch it. **Unscaled features mean the penalty silently punishes some features more than others.** This is why `prep_linear` has a `StandardScaler` and `prep_tree` does not — and why both live inside a `Pipeline`, so the scaler is fitted per fold.

In [ ]:
# TODO: Fit three linear models through `prep_linear` and compare.
#
#   models = {
#       'LinearRegression (no penalty)': LinearRegression(),
#       'Ridge (alpha=1)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
#       'Lasso (alpha=1)': Lasso(alpha=1.0, random_state=RANDOM_STATE, max_iter=5000),
#   }
#   for name, est in models.items():
#       pipe = Pipeline([('prep', prep_linear), ('model', est)])
#       pipe.fit(X_train, y_train)
#       record(regression_report(y_test, pipe.predict(X_test), name))
#
# NOTES:
#  - Lasso has no closed form, so it iterates. If you get a ConvergenceWarning,
#    raise max_iter. That warning is information, not noise -- it means the
#    optimiser stopped before settling.
#  - Set random_state on Ridge/Lasso only where your sklearn version accepts it;
#    drop it if you get a TypeError.

### 4a — What `alpha` actually does

`alpha` is the **regularisation strength** — how much you charge the model for large coefficients.

- `alpha → 0`: no penalty. The model fits the training data as hard as it can. **High variance**, overfits.
- `alpha → ∞`: the penalty dominates. All coefficients are crushed toward zero and the model predicts roughly a constant. **High bias**, underfits.

Somewhere in between is the right value, and you find it by validation, not by taste. This is the **bias–variance tradeoff** with a dial on it.

In [ ]:
# TODO: Sweep alpha for both models and plot the result.
#
#   alphas = np.logspace(-3, 3, 13)      # 0.001 ... 1000, log-spaced
#   rows = []
#   for a in alphas:
#       for name, est in [('Ridge', Ridge(alpha=a)),
#                         ('Lasso', Lasso(alpha=a, max_iter=5000))]:
#           pipe = Pipeline([('prep', prep_linear), ('model', est)])
#           pipe.fit(X_train, y_train)
#           rows.append({'alpha': a, 'model': name,
#                        'train_r2': pipe.score(X_train, y_train),
#                        'test_r2': pipe.score(X_test, y_test)})
#   sweep = pd.DataFrame(rows)
#
# Plot test_r2 against alpha on a log x-axis, one line per model, and ALSO plot
# train_r2 as a dashed line. The gap between train and test IS the variance.
#
#   for name, grp in sweep.groupby('model'):
#       plt.semilogx(grp['alpha'], grp['test_r2'], marker='o', label=f'{name} test')
#       plt.semilogx(grp['alpha'], grp['train_r2'], ls='--', alpha=0.5,
#                    label=f'{name} train')
#   plt.xlabel('alpha (regularisation strength)'); plt.ylabel('R2'); plt.legend()
#
# Then answer: where does each model peak? Which one degrades faster as alpha grows,
# and does that match what you would predict from the table above?
#
# ⚠️ alpha values are NOT comparable between Ridge and Lasso. scikit-learn's Lasso
# objective divides the error term by (2 * n_samples) while Ridge does not, so the
# same number means a different effective strength. Compare each model to ITSELF
# across alpha; never conclude "Lasso needs more regularisation" from the raw value.

In [ ]:
# TODO: The payoff -- read which features Lasso switched off.
#
# Fit a Lasso at an alpha large enough to zero out some coefficients, then inspect:
#
#   lasso_pipe = Pipeline([('prep', prep_linear),
#                          ('model', Lasso(alpha=10, max_iter=5000))])
#   lasso_pipe.fit(X_train, y_train)
#
#   names = lasso_pipe.named_steps['prep'].get_feature_names_out()
#   coefs = pd.Series(lasso_pipe.named_steps['model'].coef_, index=names)
#
#   print('features kept:  ', (coefs != 0).sum())
#   print('features zeroed:', (coefs == 0).sum())
#   coefs[coefs != 0].sort_values(key=abs, ascending=False).head(15)
#
# get_feature_names_out() matters here: after one-hot encoding your features are
# named 'nom__airline_Vistara', not 'airline'. Recovering those names is how you
# read a model.
#
# Then do the same for Ridge at the same alpha and count its exact zeros. You should
# find essentially none -- that is the L1/L2 difference, measured rather than asserted.
#
# Finally: interpret the three largest surviving coefficients as SENTENCES about
# rupees. That is the thing a tree model cannot give you.

In [ ]:
# TODO: ElasticNet -- both penalties at once (optional but worth 5 minutes).
#
#   ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=5000)
#
# `l1_ratio` mixes them: 0.0 is pure Ridge, 1.0 is pure Lasso, 0.5 is an even blend.
# It exists because the two failure modes are complementary -- Lasso alone behaves
# erratically when features are strongly correlated (it picks one of a correlated
# group almost arbitrarily, and which one can flip between data samples), while a
# little L2 stabilises that choice.
#
# Try l1_ratio in [0.1, 0.5, 0.9] and see whether any beats pure Ridge or pure Lasso.

**Lasso vs Ridge: your verdict.** *(5–7 sentences.)*

1. Which linear model scored best, and at what alpha?
2. How many features did Lasso zero out? Name two it discarded and say whether that was reasonable given your Section 2 EDA.
3. Did `LinearRegression` with no penalty do noticeably worse than the regularised versions? With ~300k rows and only ~30 one-hot columns, you have far more data than parameters — explain why that makes regularisation matter *less* here than it would on a wide, small dataset. **This is the honest answer, and it is more useful than pretending regularisation saved you.**
4. If a PM asked for "a model I can explain to a non-technical stakeholder", which of your models so far would you hand over, and what would you lose by doing so?

*(Write here)*

---
# Section 5 — Loss functions: what are you actually asking the model to minimise?

**Goal:** understand that "train a model" means "minimise a specific definition of wrong", that you *choose* that definition, and that the default is not always right.

Every model in Sections 3 and 4 minimised **squared error**, because that is the default and nobody told you otherwise. That was a choice, made silently, and it had consequences you can measure.

### Loss vs metric — keep these straight

| | **Loss function** | **Evaluation metric** |
|---|---|---|
| Used | *during* training, by the optimiser | *after* training, by you |
| Must be | differentiable (for gradient methods) | anything computable |
| Purpose | tells the model which direction to move | tells you how good the result is |
| Example | `reg:squarederror` | R², MAPE |

They do not have to match — but if your loss and your metric disagree about what "wrong" means, you are optimising for one thing and grading on another. That is worth doing deliberately, never by accident.

### The four losses that matter for regression

Let the error be `e = ŷ - y`.

| Loss | Formula | Penalises | Predicts the | Outlier behaviour |
|---|---|---|---|---|
| **Squared / L2 (MSE)** | `e²` | large errors **quadratically** | conditional **mean** | very sensitive — one bad row can dominate |
| **Absolute / L1 (MAE)** | `|e|` | errors **linearly** | conditional **median** | robust |
| **Huber** | `e²` for small `|e|`, linear beyond `δ` | both, by region | something between | robust, and smooth at zero |
| **Quantile / pinball** | `α·e` if `e<0`, `(α−1)·e` if `e>0` | over- and under-prediction **differently** | the **α-th quantile** | tunable, asymmetric |

Two consequences worth internalising:

- **MSE predicts the mean; MAE predicts the median.** That is not a detail. On the right-skewed price distribution you plotted in Section 2a, the mean sits well above the median, so these two losses produce *systematically different* predictions on the same data.
- **Huber exists because MSE and MAE each have one flaw.** MSE overreacts to outliers; MAE has a kink at zero that makes optimisation less stable and treats a ₹1 error with the same gradient as a ₹10,000 one. Huber is quadratic near zero and linear in the tails — and `δ` is where it switches.

> ### ⚠️ ReLU is an activation function, not a loss function
>
> This confusion is extremely common and worth clearing up precisely, because the two things live in different places in a model.
>
> **ReLU** is `f(x) = max(0, x)`. It is an **activation function**: it sits *inside* a neural network, transforming one layer's output before it reaches the next. It takes one number and returns one number. It has no idea what the true label is, so **it cannot be a loss** — a loss must compare a prediction to a truth.
>
> ```
> input -> [linear layer] -> ReLU -> [linear layer] -> ReLU -> output -> LOSS(output, truth)
>                             ^^^^                     ^^^^              ^^^^
>                        activations: shape the model   |          loss: scores the model
> ```
>
> **But `max(0, ·)` genuinely does appear inside loss functions**, which is probably where the confusion comes from — and this part is useful:
>
> - **Hinge loss** (classification, SVMs): `max(0, 1 − y·ŷ)`. Zero penalty once you are correct by a margin, linear penalty otherwise. That is literally a ReLU applied to the margin.
> - **Pinball / quantile loss** (regression, below): `max(α·e, (α−1)·e)` — two ReLU-shaped arms joined at zero, with **different slopes** on each side. The asymmetry of those slopes is the whole mechanism.
>
> So: ReLU is a **building block** of asymmetric and margin-based losses, not a loss itself. If a teammate says "let's use ReLU as our loss", the useful reply is "do you mean a hinge or a quantile loss?" — because that is almost always the idea underneath.

### 5a — Quantile loss, and why it is the interesting one for pricing

Every loss above is **symmetric**: over-predicting by ₹500 costs exactly what under-predicting by ₹500 costs.

**Pricing is not symmetric.** Consider the two ways to be wrong:

- **Quote too low.** You sell the seat, but you left money on the table — a loss of margin on one booking.
- **Quote too high.** The customer books a competitor. You lose the entire sale, and possibly the customer.

Those costs are not equal, and which is worse depends on the business situation — a nearly-full flight versus one departing half-empty. A symmetric loss cannot express any of that. **Quantile loss can**, because `α` sets the asymmetry:

- `α = 0.5` → symmetric → predicts the **median** (this is exactly MAE)
- `α = 0.9` → under-prediction punished 9× harder → predicts the **90th percentile**, a deliberately high quote
- `α = 0.1` → over-prediction punished 9× harder → predicts the **10th percentile**, a deliberately low quote

Fit three of them and you get a **prediction interval** rather than a single number — "this fare is most likely ₹6,400, and we are fairly confident it is between ₹4,900 and ₹8,800". For a pricing recommendation, that interval is far more useful than a point estimate, because it tells the user how much to trust it.

In [ ]:
# TODO: Train XGBoost with three different objectives and compare.
#
#   objectives = {
#       'XGB  MSE  (reg:squarederror)':   dict(objective='reg:squarederror'),
#       'XGB  MAE  (reg:absoluteerror)':  dict(objective='reg:absoluteerror'),
#       'XGB  Huber (pseudohuber)':       dict(objective='reg:pseudohubererror'),
#   }
#   for name, kw in objectives.items():
#       pipe = Pipeline([('prep', prep_tree),
#                        ('model', XGBRegressor(n_estimators=300, max_depth=6,
#                                               learning_rate=0.1,
#                                               random_state=RANDOM_STATE,
#                                               n_jobs=-1, **kw))])
#       pipe.fit(X_train, y_train)
#       record(regression_report(y_test, pipe.predict(X_test), name))
#
# ⚠️ VERSION CHECK: 'reg:absoluteerror' needs xgboost >= 1.7. Print
# xgboost.__version__; if yours is older, substitute the sklearn equivalent:
#   GradientBoostingRegressor(loss='absolute_error', n_estimators=150,
#                             random_state=RANDOM_STATE)
# (slower, so drop n_estimators) -- and note the substitution in your PR.
#
# READING THE RESULT -- this is the part that matters. Do not just look at R2:
#   - the MSE-trained model should win on RMSE
#   - the MAE-trained model should win on MAE and probably MAPE%
# That is not a coincidence or a fluke. Each model wins on the metric matching the
# loss it was trained to minimise. Write that down; it is the lesson of the section.

In [ ]:
# TODO: Fit three quantile models and build a prediction interval.
#
# sklearn's GradientBoostingRegressor supports quantile loss in every recent
# version, so we use it here for portability. (xgboost >= 2.0 can do this with
# objective='reg:quantileerror' and quantile_alpha -- use that if you have it.)
#
#   quantiles = [0.1, 0.5, 0.9]
#   q_preds = {}
#   for q in quantiles:
#       pipe = Pipeline([('prep', prep_tree),
#                        ('model', GradientBoostingRegressor(
#                            loss='quantile', alpha=q, n_estimators=150,
#                            max_depth=5, random_state=RANDOM_STATE))])
#       pipe.fit(X_train, y_train)
#       q_preds[q] = pipe.predict(X_test)
#       print(f'q={q} fitted')
#
# NOTE: three separate fits. Quantile regression gives you one model PER quantile --
# there is no single model that emits all three. Budget the time accordingly.
#
# 1. COVERAGE CHECK -- the honest test of an interval. Roughly 80% of true prices
#    should fall between the 0.1 and 0.9 predictions. If you get 95% the interval is
#    too wide to be useful; if 50%, it is lying about its own confidence:
#      inside = ((y_test >= q_preds[0.1]) & (y_test <= q_preds[0.9])).mean()
#      print(f'coverage: {100 * inside:.1f}%  (target ~80%)')
#
# 2. Plot the interval for ~40 test rows sorted by predicted median, using
#    plt.fill_between for the band and plt.scatter for the actuals.
#
# 3. Score each with mean_pinball_loss(y_test, pred, alpha=q) -- the metric that
#    matches the loss.

**Loss functions: your verdict.** *(5–7 sentences.)*

1. Which objective won on which metric? State the pattern you observed in one sentence.
2. Given the price distribution from Section 2a — two populations, strongly right-skewed overall — which loss do you think is the right default for *this* dataset, and why?
3. What was your 80% interval's actual coverage? If it missed the target, is the model over- or under-confident?
4. **The business question.** For a fare-recommendation tool, is it worse to suggest a price that is too high or too low? Pick a side, commit to it, and say which `α` you would ship. There is no neutral answer — the symmetric default *is* a choice, it is just an unexamined one.

*(Write here)*

---
# Section 6 — XGBoost vs Random Forest vs ARIMA

**Goal:** choose a model family on the basis of what the problem is, not what the leaderboard says.

These three are not three contestants in one race. **Two of them answer the same question in different ways, and the third answers a different question entirely** — recognising that is the actual skill in this section.

| Model | Family | What it needs | What it predicts |
|---|---|---|---|
| **Random Forest** | bagged trees | rows of features, any order | one target value **per row** |
| **XGBoost** | boosted trees | rows of features, any order | one target value **per row** |
| **ARIMA** | time series | one **ordered sequence**, equally spaced | the **next values** of that sequence |

Random Forest and XGBoost are interchangeable in their interface: both take `X, y` and predict a price for an individual ticket. ARIMA cannot do that at all — it does not take features. It takes a *sequence* and extends it.

---

### 6a — Bagging vs boosting: how RF and XGBoost actually differ

Both build many decision trees. **How** they build them is close to opposite.

| | **Random Forest (bagging)** | **XGBoost (boosting)** |
|---|---|---|
| Trees are built | **independently**, all in parallel | **sequentially**, each one after the last |
| Each tree is trained on | a bootstrap sample + a random feature subset | the **errors (gradients)** the ensemble still has |
| Individual trees are | deep, overfit, **decorrelated** | shallow, weak learners |
| Combined by | **averaging** | **adding** (scaled by the learning rate) |
| Mainly reduces | **variance** | **bias** |
| More trees → | never really worse, just slower | **can overfit**; needs `learning_rate` / early stopping |
| Tuning burden | low — works well out of the box | higher — more knobs that interact |
| Parallelises | trivially (trees are independent) | within a tree, not across them |

The intuition: **bagging** builds many different experts and averages away their individual mistakes. **Boosting** builds one expert, finds what it gets wrong, and hires a specialist for exactly that — repeatedly.

This explains their failure modes. A Random Forest rarely blows up but often plateaus slightly below a well-tuned booster. XGBoost usually wins *after tuning* and overfits cheerfully if you leave `n_estimators` high and `learning_rate` high.

In [ ]:
# TODO: Head-to-head, same pipeline, same split, with timings.
#
#   contenders = {
#       'Dummy (mean)': DummyRegressor(strategy='mean'),
#       'RandomForest': RandomForestRegressor(
#           n_estimators=100, max_depth=None, min_samples_leaf=2,
#           random_state=RANDOM_STATE, n_jobs=-1),
#       'XGBoost': XGBRegressor(
#           n_estimators=300, max_depth=6, learning_rate=0.1,
#           subsample=0.8, colsample_bytree=0.8,
#           random_state=RANDOM_STATE, n_jobs=-1),
#   }
#   for name, est in contenders.items():
#       pipe = Pipeline([('prep', prep_tree), ('model', est)])
#       t0 = time.time()
#       pipe.fit(X_train, y_train)
#       rep = regression_report(y_test, pipe.predict(X_test), name)
#       rep['fit_seconds'] = round(time.time() - t0, 1)
#       record(rep)
#
# ⚠️ TIMING AND MEMORY: an unbounded Random Forest on ~240k training rows grows
# very large trees and can take several minutes and a lot of RAM. If it struggles:
#   - set max_depth=20, or
#   - subsample the training set for this cell and note it in your PR.
# That is itself a finding worth reporting: "RF needed 10x the fit time for a
# comparable score" is a legitimate reason to choose the other model.
#
# ALWAYS include the Dummy. "R2 = 0.95" means nothing until you know the floor.

**RF vs XGBoost.** *(4–6 sentences.)*

- Which won, and by how much in **MAE rupees** (not R² — rupees is the version a stakeholder understands)?
- How did fit times compare? If one took 10× longer for 1% more R², which would you ship, and what does that depend on?
- How far above the `DummyRegressor` is each? That gap is how much your features are actually worth.
- Which would you rather hand to a teammate who has never tuned a model, and why?

*(Write here)*

### 6b — ARIMA: the right tool for a question we barely have

**ARIMA** = **A**uto**R**egressive **I**ntegrated **M**oving **A**verage, written `ARIMA(p, d, q)`:

- **AR(p)** — regress the value on its own previous `p` values. "Today looks like yesterday."
- **I(d)** — **difference** the series `d` times to make it **stationary** (constant mean/variance over time). A series that trends upward forever cannot be modelled directly; its *changes* usually can.
- **MA(q)** — regress on the previous `q` forecast *errors*. Lets recent shocks decay.

ARIMA is the standard tool for forecasting a single quantity through time: monthly revenue, daily passengers, weekly bookings.

### So does it apply to this dataset?

Be honest about this, because it is the real lesson:

**Mostly, no — and that is a legitimate finding.** ARIMA needs a series indexed by time, and this dataset **has no dates**. There is no departure date, no booking date, no timestamp anywhere. Without a calendar axis there is no seasonality, no trend over real time, no day-of-week effect. The dataset is a cross-section: 300,000 independent snapshots with no temporal order among them.

**But there is one real sequence in here**, and you plotted it in Section 2c: **mean price by `days_left`**. Days-to-departure is an ordered, equally-spaced index, so the booking curve is a genuine series of ~49 points — and forecasting the booking curve is a real revenue-management task. So we apply ARIMA to the thing it can actually model, and then we are precise about what we did and did not learn.

> **The transferable lesson:** your first model-selection question is never "which model scores best?" It is **"what shape is my data, and what shape does this model require?"** A model that needs a time index will not acquire one because you want it to. Picking the wrong family is a mistake no amount of hyperparameter tuning can repair.

In [ ]:
# TODO: Build the booking-curve series and check it for stationarity.
#
#   from statsmodels.tsa.stattools import adfuller
#
#   # Sort DESCENDING by days_left, so position 0 is furthest from departure and
#   # the last position is the day before. Time now flows with the index.
#   curve = clean.groupby('days_left')['price'].mean().sort_index(ascending=False)
#   series = curve.reset_index(drop=True)
#   print(f'{len(series)} points')
#
#   # Augmented Dickey-Fuller test. H0 = "the series is NON-stationary".
#   # p < 0.05 lets you reject H0, i.e. evidence it IS stationary.
#   stat, p, *_ = adfuller(series)
#   print(f'ADF p-value (raw):        {p:.4f}')
#   stat_d, p_d, *_ = adfuller(series.diff().dropna())
#   print(f'ADF p-value (differenced): {p_d:.4f}')
#
# Plot the series and its first difference side by side.
#
# Then answer: is the raw curve stationary? Does differencing once fix it? That
# answer IS your `d` parameter -- if raw is non-stationary and differenced is
# stationary, use d=1.

In [ ]:
# TODO: Fit ARIMA and forecast the last 9 points, against a naive baseline.
#
#   from statsmodels.tsa.arima.model import ARIMA
#
#   train_s, test_s = series[:-9], series[-9:]
#
#   arima = ARIMA(train_s, order=(2, 1, 2)).fit()   # use YOUR d from above
#   print(arima.summary())
#   fc = arima.forecast(steps=9)
#
#   # THE BASELINE THAT KEEPS YOU HONEST: carry the last observed value forward.
#   # On a short series a naive forecast is a genuinely hard baseline to beat, and
#   # plenty of published forecasting results quietly fail to beat it.
#   naive = np.repeat(train_s.iloc[-1], 9)
#
#   print('ARIMA MAE:', mean_absolute_error(test_s, fc))
#   print('Naive MAE:', mean_absolute_error(test_s, naive))
#
# Plot: the training series, the actual held-out tail, the ARIMA forecast, and the
# naive forecast, all on one axes.
#
# Try two or three other (p, d, q) orders -- e.g. (1,1,0), (0,1,1), (3,1,1) -- and
# compare AIC from arima.summary(). Lower AIC is better; it trades fit against
# parameter count. Do not read too much into it with 40 training points.
#
# ⚠️ Expect convergence warnings. 40 points is a very small sample for ARIMA, and
# saying so in your write-up is more valuable than hiding it.

**ARIMA: the verdict.** *(5–7 sentences. This is the most important written answer in the notebook.)*

1. Did ARIMA beat the naive last-value forecast? By how much? If it did not, say so plainly — a negative result reported honestly is worth more than a flattering one.
2. **What question did this ARIMA model answer?** Be precise. It did *not* price a flight. Write down what it actually produces and who could use it.
3. **What did you give up to use it?** You aggregated 300,000 rows into ~49 points. Name what was thrown away. Can this model tell you anything about airline, class, route or stops?
4. Would you include ARIMA in the deliverable to American Airlines? If so, in what role — alongside XGBoost rather than instead of it? If not, what would have to change about the dataset for it to earn a place?

*(Write here)*

> **For the curious: SARIMAX.** The `statsmodels` family does extend to seasonality and features: `SARIMAX` adds a seasonal component and an `exog=` argument for external regressors. With real departure dates you could model weekly and holiday seasonality and feed in fuel price or capacity — which is much closer to how an airline forecasts demand. We cannot, because this dataset has no dates. Worth knowing the door exists.

### 6c — The model-selection decision table

Fill this in from your own numbers. This table, not any single score, is the deliverable for Section 6.

| Model | Best MAE (INR) | R² | Fit time | Interpretable? | When would you choose it? |
|---|---|---|---|---|---|
| Dummy (mean) | | | | — | never; it is the floor |
| Ridge | | | | yes — coefficients | |
| Lasso | | | | yes — sparse coefficients | |
| Random Forest | | | | no | |
| XGBoost | | | | no | |
| ARIMA | n/a — different target | n/a | | partly | |

**The one you would ship, and the three reasons why:**

*(Not just accuracy. Weigh fit time, tuning burden, interpretability, and what the stakeholder actually needs.)*

---
# Section 7 — Defend your choice

### 7a — The full scoreboard

In [ ]:
# TODO: Print the complete scoreboard and sort it by MAE.
#
#   show_scoreboard().sort_values('MAE')
#
# Then write the one-line summary a PM would actually read:
#   "Best model: ____, MAE ₹____ (±__ across CV folds), vs ₹____ for the dummy."

### 7b — What this means for American Airlines

We are modelling **Indian domestic fares scraped from EaseMyTrip in 2022**. American Airlines is a different carrier in a different market in a different year. Be precise about what transfers.

**What transfers — the structure of the problem and the method.** Airline revenue management has the same shape everywhere: fares move with the booking horizon, cabin class dominates price, route density and competition set the base level, and time-of-day and connections differentiate the product. Your model-selection reasoning transfers completely: tree ensembles for per-ticket prediction, asymmetric loss where over- and under-pricing cost differently, a time-series model for forecasting an aggregate curve through real time.

**What does not transfer:** nearly all the specifics. Different regulatory regime, competitive set, cost base, currency, and demand seasonality — and 2022 was still distorted by post-pandemic recovery.

**What the dataset is missing that real pricing requires:**

| Missing | Why it matters |
|---|---|
| **Seat inventory / load factor** | Revenue management is fundamentally about *remaining seats*. We have no capacity data at all — the largest single gap. |
| **Departure dates** | We have `days_left` but not *when*. No seasonality, no holidays, no day-of-week. **This is also precisely why ARIMA had so little to work with.** |
| **Competitor fares** | Fares are set against what everyone else charges on the route, in near-real-time. |
| **Fare class / fare basis** | "Economy" spans a dozen booking classes with different prices and restrictions; our `class` flattens all of it to two values. |
| **Demand signals** | Searches, booking pace, events — the leading indicators pricing teams actually watch. |

### 7c — Reflection

Answer each in the cell below it, in your own words. A PM will ask these in your PR review, so treat it as practice for that conversation.

---

**Q1 — The honest number.** You produced several R² values for the same task. List them with one line each on what changed, then name the single number you would put in front of American Airlines and defend it in two sentences.

**Answer Q1:** *(Write here)*

**Q2 — Model family.** You would ship one of Ridge / Lasso / Random Forest / XGBoost. Name it, and give **three** reasons that are not "it had the best score" — e.g. fit time, tuning burden, interpretability, robustness, what the stakeholder needs. Then name the strongest argument *against* your choice.

**Answer Q2:** *(Write here)*

**Q3 — Lasso vs Ridge.** Explain to a teammate who has not done this deliverable why L1 produces exact zeros and L2 does not. Then say which you would use on this dataset and why — including the possibility that neither penalty mattered much here, and what about this dataset makes that the case.

**Answer Q3:** *(Write here)*

**Q4 — Loss functions.** Your Deliverable 1 model minimised squared error because that is the default. Having seen the comparison: was that the right default for this data? Which loss would you train on now, and which metric would you report? If those two differ, justify the mismatch.

**Answer Q4:** *(Write here)*

**Q5 — ARIMA and problem shape.** Someone proposes ARIMA for predicting individual ticket prices. Explain in three sentences why that does not work, and what they would need to change about the data or the question for a time-series model to be the right call.

**Answer Q5:** *(Write here)*

**Q6 — Leakage.** Explain leakage to a teammate using an example from your own notebook run (not one from the text). Then name the one habit from this notebook you will carry into Deliverable 3 to prevent it.

**Answer Q6:** *(Write here)*

**Q7 — Team git.** Deliverable 1 was `deliverable1/<your-name>` → `main`. Deliverable 2 is `team-##/<your-name>` → `team-##/main` → `main`. What does the extra layer buy a project with 70+ contributors? Then describe something that actually went wrong or felt awkward in your team's git this week — a real friction point is worth more here than a clean answer.

**Answer Q7:** *(Write here)*

---
## Before you submit — checklist

**The notebook**

1. **Kernel → Restart & Run All.** It must run top-to-bottom with no errors. A notebook that only works in the order you happened to click cells is not reproducible.
2. Every `# TODO` cell is filled in.
3. Every markdown answer prompt has a real answer. "*(Write here)*" left anywhere is an incomplete deliverable.
4. The three summary tables are filled in: the EDA decisions (2e), the model-selection table (6c), and the scoreboard (7a).
5. **Kernel → Restart & Clear Output** before committing. Keeps the diff reviewable and prevents most notebook merge conflicts.

**Git hygiene**

6. `git status` shows **no** `.csv`. If it does, stop — do not commit, and check [`.gitignore`](../.gitignore).
7. You are on `team-##/<your-name>` — not `team-##/main`, and definitely not `main`. Check with `git branch --show-current`.

```sh
git add deliverable2/flight_price_modeling.ipynb
git commit -m "deliverable2: EDA, leakage fixes, model selection"
git push
```

8. Open a PR from `team-##/<your-name>` into **`team-##/main`**. GitHub defaults the base to `main` — change it by hand.
9. Request review from the teammate assigned to you in [`README.md`](README.md), and **review the PR assigned to you**. Reviewing is half the job.
10. Once every member's PR has merged into `team-##/main`, the team fills in [`TEAM_REPORT.md`](TEAM_REPORT.md) together and a PM reviews `team-##/main` → `main`.

---

You arrived with one model and one score. You are leaving with five models, four loss functions, an honest number, and — most importantly — a reason for every choice.

In Deliverable 3 we tune the winner: hyperparameter search over the pipeline you built, learning curves to distinguish a data problem from a model problem, and error analysis by segment to find out *where* it fails rather than just how much. 🛫